In [1]:
#1 — Imports
import os
import sys
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from skimage.color import rgb2lab, lab2rgb
import torch
import torch.nn as nn

sys.path.append(os.path.expanduser("~/colorizer_gan/notebooks"))
from model import Generator

In [2]:
# Paths
MODEL_PATH  = os.path.expanduser("~/colorizer_gan/models/checkpoint_epoch_100.pth")
OUTPUT_PATH = os.path.expanduser("~/colorizer_gan/outputs/inference")
IMAGE_SIZE  = 256
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(OUTPUT_PATH, exist_ok=True)

def load_generator(model_path, device):
    G = Generator().to(device)
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    G.load_state_dict(checkpoint["G_state"])
    G.eval()
    print(f"Generator loaded from {model_path}")
    print(f"   Trained for {checkpoint['epoch']} epochs")
    return G

G = load_generator(MODEL_PATH, DEVICE)
print(f"\nDevice: {DEVICE}")

Generator loaded from /home/61447093s/colorizer_gan/models/checkpoint_epoch_100.pth
   Trained for 100 epochs

Device: cuda


In [3]:
#3 — picture detetion
def detect_image_type(image_path):
    """
    Detect if image is grayscale or color
    Returns: 'grayscale' or 'color'
    """
    img    = Image.open(image_path).convert("RGB")
    img_np = np.array(img).astype("float32")

    r = img_np[:, :, 0]
    g = img_np[:, :, 1]
    b = img_np[:, :, 2]

    # Check if all channels are equal within tolerance
    rg_diff = np.mean(np.abs(r - g))
    rb_diff = np.mean(np.abs(r - b))
    gb_diff = np.mean(np.abs(g - b))

    avg_diff = (rg_diff + rb_diff + gb_diff) / 3.0

    if avg_diff < 2.0:      # threshold — channels nearly identical
        result = "grayscale"
    else:
        result = "color"

    print(f" Autodetect: image is {result} (avg channel diff: {avg_diff:.2f})")
    return result

In [4]:
#4 — preprocessing Function
def preprocess(image_path, image_size=256):
    """
    Load and preprocess image for model input
    Returns: L tensor, original numpy image
    """
    img    = Image.open(image_path).convert("RGB")
    img    = img.resize((image_size, image_size), Image.BICUBIC)
    img_np = np.array(img) / 255.0

    # Convert to LAB
    lab = rgb2lab(img_np).astype("float32")

    # Extract and normalize L channel
    L = (lab[:, :, 0] / 50.0) - 1.0
    L = torch.tensor(L).unsqueeze(0).unsqueeze(0)  # (1, 1, H, W)

    return L, img_np

In [5]:
#5 — colorize Function
def colorize(image_path, G, device, image_size=256):
    """
    Colorize a grayscale image using the trained generator
    Returns: grayscale numpy array, colorized numpy array
    """
    L_tensor, img_np = preprocess(image_path, image_size)
    L_tensor = L_tensor.to(device)

    with torch.no_grad():
        AB_pred = G(L_tensor).cpu().squeeze(0)  # (2, H, W)

    # Denormalize
    L_np       = (L_tensor.cpu().squeeze().numpy() + 1.0) * 50.0
    AB_np      = AB_pred.permute(1, 2, 0).numpy() * 128.0

    # Reconstruct LAB image
    lab_out          = np.zeros((image_size, image_size, 3), dtype="float32")
    lab_out[:, :, 0] = L_np
    lab_out[:, :, 1:]= AB_np

    # Convert to RGB
    rgb_colorized = np.clip(lab2rgb(lab_out), 0, 1)

    # Grayscale version for display
    gray = np.stack([L_np / 100.0] * 3, axis=-1)
    gray = np.clip(gray, 0, 1)

    return gray, rgb_colorized

In [6]:
#6 — gray scale function
def to_grayscale(image_path, image_size=256):
    """
    Convert a color image to grayscale
    Returns: color numpy array, grayscale numpy array
    """
    img    = Image.open(image_path).convert("RGB")
    img    = img.resize((image_size, image_size), Image.BICUBIC)
    img_np = np.array(img) / 255.0

    # Convert to LAB and extract L
    lab    = rgb2lab(img_np).astype("float32")
    L_np   = lab[:, :, 0] / 100.0
    gray   = np.stack([L_np] * 3, axis=-1)
    gray   = np.clip(gray, 0, 1)

    return img_np, gray

In [7]:
#7 — Run Inference on Multiple Images (batch)
def auto_process(image_path, G, device, image_size=256, mode="auto"):
    """
    Automatically detect and process image
    mode: 'auto', 'colorize', 'grayscale'
    Returns: input_display, output_display, detected_mode
    """
    if mode == "auto":
        detected = detect_image_type(image_path)
    else:
        detected = mode

    if detected == "grayscale":
        print(f"Mode: Colorizing...")
        input_img, output_img = colorize(image_path, G, device, image_size)
        mode_label = "Grayscale → Color"
    else:
        print(f" Mode: Converting to grayscale...")
        input_img, output_img = to_grayscale(image_path, image_size)
        mode_label = "Color → Grayscale"

    return input_img, output_img, mode_label

In [8]:
#8 single image inference
def run_single(image_path, G, device, mode="auto", save=True):
    """
    Run inference on a single image and save result
    """
    filename   = os.path.splitext(os.path.basename(image_path))[0]
    input_img, output_img, mode_label = auto_process(
        image_path, G, device, IMAGE_SIZE, mode
    )

    # Save output
    if save:
        output_pil  = Image.fromarray((output_img * 255).astype(np.uint8))
        save_path   = os.path.join(OUTPUT_PATH, f"{filename}_result.png")
        output_pil.save(save_path)
        print(f" Result saved to {save_path}")

    # Display side by side
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    fig.suptitle(f"Mode: {mode_label}", fontsize=13, fontweight="bold")

    axes[0].imshow(input_img)
    axes[0].set_title("Input")
    axes[0].axis("off")

    axes[1].imshow(output_img)
    axes[1].set_title("Output")
    axes[1].axis("off")

    plt.tight_layout()
    plot_path = os.path.join(OUTPUT_PATH, f"{filename}_comparison.png")
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f" Comparison saved to {plot_path}")

    return output_img

# ---- Test on a sample image from validation set ----
VAL_PATH   = "/mnt/hdd/colorizer_gan/datasets/coco/val/val2017"
test_image = os.path.join(VAL_PATH, os.listdir(VAL_PATH)[0])
print(f"Testing on: {test_image}")

run_single(test_image, G, DEVICE, mode="auto")

Testing on: /mnt/hdd/colorizer_gan/datasets/coco/val/val2017/000000520659.jpg
 Autodetect: image is color (avg channel diff: 17.48)
 Mode: Converting to grayscale...
 Result saved to /home/61447093s/colorizer_gan/outputs/inference/000000520659_result.png
 Comparison saved to /home/61447093s/colorizer_gan/outputs/inference/000000520659_comparison.png


array([[[0.03805506, 0.03805506, 0.03805506],
        [0.03155973, 0.03155973, 0.03155973],
        [0.04098739, 0.04098739, 0.04098739],
        ...,
        [0.40021503, 0.40021503, 0.40021503],
        [0.2184223 , 0.2184223 , 0.2184223 ],
        [0.23834038, 0.23834038, 0.23834038]],

       [[0.03176264, 0.03176264, 0.03176264],
        [0.02884503, 0.02884503, 0.02884503],
        [0.03440712, 0.03440712, 0.03440712],
        ...,
        [0.32478583, 0.32478583, 0.32478583],
        [0.22861731, 0.22861731, 0.22861731],
        [0.21575709, 0.21575709, 0.21575709]],

       [[0.02884503, 0.02884503, 0.02884503],
        [0.03782806, 0.03782806, 0.03782806],
        [0.03440712, 0.03440712, 0.03440712],
        ...,
        [0.22859374, 0.22859374, 0.22859374],
        [0.23310193, 0.23310193, 0.23310193],
        [0.22702634, 0.22702634, 0.22702634]],

       ...,

       [[0.3697662 , 0.3697662 , 0.3697662 ],
        [0.38268104, 0.38268104, 0.38268104],
        [0.36111343, 0

In [ ]:
#batch inference
def run_batch(image_paths, G, device, mode="auto", n_display=6):
    """
    Run inference on multiple images and save comparison grid
    """
    n       = min(n_display, len(image_paths))
    fig, axes = plt.subplots(n, 2, figsize=(10, n * 4))
    fig.suptitle("Batch Inference Results", fontsize=14, fontweight="bold")

    for i, image_path in enumerate(image_paths[:n]):
        input_img, output_img, mode_label = auto_process(
            image_path, G, device, IMAGE_SIZE, mode
        )

        axes[i, 0].imshow(input_img)
        axes[i, 0].set_title(f"Input ({mode_label.split('→')[0].strip()})")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(output_img)
        axes[i, 1].set_title(f"Output ({mode_label.split('→')[1].strip()})")
        axes[i, 1].axis("off")

    plt.tight_layout()
    save_path = os.path.join(OUTPUT_PATH, "batch_results.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f" Batch results saved to {save_path}")

# ---- Test on 6 validation images ----
val_files = [
    os.path.join(VAL_PATH, f)
    for f in os.listdir(VAL_PATH)[:6]
]

run_batch(val_files, G, DEVICE, mode="auto")

In [ ]:
#report comparison grid chart
def report_grid(image_paths, G, device, mode="colorize", n=4):
    """
    Generate a clean comparison grid for the report
    Shows: Input | Grayscale | Colorized
    """
    fig, axes = plt.subplots(n, 3, figsize=(14, n * 4))
    fig.suptitle(
        "Colorization Results — Grayscale Input vs Model Output vs Ground Truth",
        fontsize=13, fontweight="bold"
    )

    col_titles = ["Ground Truth", "Grayscale Input", "Colorized Output"]
    for ax, title in zip(axes[0], col_titles):
        ax.set_title(title, fontsize=12, fontweight="bold")

    for i, image_path in enumerate(image_paths[:n]):
        # Load original color image
        img    = Image.open(image_path).convert("RGB")
        img    = img.resize((IMAGE_SIZE, IMAGE_SIZE), Image.BICUBIC)
        img_np = np.array(img) / 255.0

        # Get grayscale and colorized
        gray, colorized = colorize(image_path, G, device, IMAGE_SIZE)

        axes[i, 0].imshow(img_np)
        axes[i, 1].imshow(gray)
        axes[i, 2].imshow(colorized)

        for ax in axes[i]:
            ax.axis("off")

    plt.tight_layout()
    save_path = os.path.join(OUTPUT_PATH, "report_grid.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Report grid saved to {save_path}")

# ---- Generate report grid from validation set ----
val_files = [
    os.path.join(VAL_PATH, f)
    for f in os.listdir(VAL_PATH)[:4]
]

report_grid(val_files, G, DEVICE, n=4)